# 5.2 KV cache 与 prefill/decode / KV cache, prefill and decode

生成时怎样避免重复计算，推理为什么分成两个阶段，以及首 token 延迟和生成速度各由什么决定。

How repeated computation is avoided during generation, why inference has two phases, and what determines time to first token and generation speed.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 给 mini-GPT 加上 KV cache / Add a KV cache to mini-GPT

沿用 2.2 的 `MultiHeadAttention` 和 2.5 的 `MiniGPT`，实现一次只处理一个 token 的前向计算，并把先前位置的键和值保存在缓存里。测试会检查每个位置的 logits 与完整前向计算一致，带缓存的生成与不带缓存的生成结果相同，并统计两种方式各处理了多少个位置。

Reusing `MultiHeadAttention` from 2.2 and `MiniGPT` from 2.5, implement a forward pass that processes one token at a time and keeps the keys and values of earlier positions in a cache. The tests check that the logits at every position agree with the full forward pass, that generation with the cache equals generation without it, and count how many positions each way processes.

In [ ]:
# 来自练习 m22_mha 的参考实现，本题会用到 / reference code from exercise m22_mha, used below
import math
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.d_head = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                                   # each (B, T, d)
        split = lambda t: t.view(B, T, self.n_head, self.d_head).transpose(1, 2)  # (B, h, T, d_head)
        q, k, v = split(q), split(k), split(v)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)                # (B, h, T, T)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        out = weights @ v                                                        # (B, h, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, d)                     # glue the heads back together
        return self.proj(out), weights


def mha_param_count(d_model, n_head):
    return (3 * d_model * d_model + 3 * d_model) + (d_model * d_model + d_model)  # independent of n_head

In [ ]:
# 来自练习 m25_minigpt 的参考实现，本题会用到 / reference code from exercise m25_minigpt, used below
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    """A Pre-Norm Transformer block."""

    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)[0]
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size=32, d_model=64, n_head=4, n_layer=2):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        """idx: (B, T) token ids. Returns (logits (B, T, V), loss or None)."""
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T))
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
        for block in self.blocks:
            x = block(x, mask)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, generator=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = torch.softmax(logits[:, -1], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1, generator=generator)], dim=1)
        return idx


def train(model, data, steps=300, batch_size=32, lr=3e-3, seed=0):
    """Train on a 1-D tensor of token ids. Returns the list of loss values. Nothing to change here."""
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        ix = torch.randint(len(data) - model.block_size - 1, (batch_size,), generator=g)
        x = torch.stack([data[i:i + model.block_size] for i in ix])
        y = torch.stack([data[i + 1:i + model.block_size + 1] for i in ix])     # the same text, shifted by one
        _, loss = model(x, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

In [ ]:
import math
import torch

# MultiHeadAttention (module 2.2) and MiniGPT (module 2.5) are already defined above this cell.
# MiniGPT.forward recomputes every position at every step. Here the model is run one token at a time,
# and the keys and values of earlier tokens are kept in a cache.
#
# The cache of one layer is a tuple (k, v), each of shape (B, n_head, t, d_head), or None when empty.


def attention_step(attn, x_new, cache):
    """Attention for ONE new position, reusing the keys and values of earlier positions.

    attn:   a MultiHeadAttention module (use attn.qkv, attn.proj, attn.n_head, attn.d_head)
    x_new:  (B, 1, d), the input of the new position only
    cache:  (k, v) of the earlier positions, or None
    returns (out (B, 1, d), new_cache)
    """
    # TODO 1: q, k, v of the new position: attn.qkv(x_new).chunk(3, dim=-1), then split into heads -> (B, h, 1, d_head)
    # TODO 2: if there is a cache, append the new k and v to it along the position dimension (dim=2)
    # TODO 3: scores = q @ k^T / sqrt(d_head)  -> (B, h, 1, t+1). No causal mask is needed. Why?
    # TODO 4: softmax, weighted sum of v, glue the heads together, attn.proj
    raise NotImplementedError


@torch.no_grad()
def forward_step(model, token, pos, caches):
    """One step of MiniGPT on a single token.

    token:  (B, 1) token ids of the new position
    pos:    its position index (int)
    caches: a list with one cache per block
    returns (logits (B, V) for the next token, new list of caches)
    """
    # TODO 5: x = token embedding + position embedding of `pos`
    # TODO 6: for every block (Pre-Norm, as in MiniGPT):
    #             a, cache = attention_step(block.attn, block.ln1(x), cache);  x = x + a
    #             x = x + block.mlp(block.ln2(x))
    # TODO 7: final LayerNorm and head
    raise NotImplementedError


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """Greedy generation with the cache. idx: (B, T). Returns (B, T + max_new_tokens).

    Feed the prompt one token at a time to fill the cache, then generate. Every token passes
    through the model exactly once.
    """
    # TODO 8
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

torch.manual_seed(0)
V = 50
model = MiniGPT(V, block_size=32, d_model=64, n_head=4, n_layer=3).eval()
idx = torch.randint(0, V, (2, 12))

# 1. Step by step with the cache gives the same logits as the full forward pass, at every position.
with torch.no_grad():
    full, _ = model(idx)                                             # (2, 12, V)
caches = [None] * len(model.blocks)
for t in range(idx.shape[1]):
    step_logits, caches = forward_step(model, idx[:, t:t + 1], t, caches)
    assert step_logits.shape == (2, V)
    assert torch.allclose(step_logits, full[:, t], atol=1e-5), f"position {t} differs from the full forward pass"

# 2. The cache holds one key and one value per layer, head and position.
assert len(caches) == 3
for k, v in caches:
    assert k.shape == (2, 4, 12, 16) and v.shape == (2, 4, 12, 16)

# 3. Cached greedy generation equals greedy generation that recomputes everything.
def generate_plain(model, idx, n):
    for _ in range(n):
        with torch.no_grad():
            logits, _ = model(idx)
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx

out = generate_cached(model, idx, 15)
assert out.shape == (2, 27) and torch.equal(out, generate_plain(model, idx, 15))

# 4. Count the work: with the cache every token is embedded exactly once.
calls = {"plain": 0, "cached": 0}
hook = None
def counter(name):
    def f(module, inp, out):
        calls[name] += inp[0].numel()
    return f
h = model.tok_emb.register_forward_hook(counter("plain")); generate_plain(model, idx[:1], 15); h.remove()
h = model.tok_emb.register_forward_hook(counter("cached")); generate_cached(model, idx[:1], 15); h.remove()
assert calls["cached"] == 12 + 15 - 1, calls           # every token once; the last generated token is never fed back
assert calls["plain"] == sum(range(12, 27)), calls     # 12 + 13 + ... + 26
print(f"m52_kvcache: all tests passed (token positions processed: {calls['plain']} without the cache, {calls['cached']} with it)")

<details><summary><b>提示 1 / Hint 1</b></summary>

缓存的形状是 `(B, n_head, t, d_head)`，新的键和值沿 `dim=2` 拼接。

The cache has shape `(B, n_head, t, d_head)`, and new keys and values are concatenated along `dim=2`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

这里不需要因果掩码，因为缓存中只有当前位置和它之前的位置，没有未来的位置可以屏蔽。

No causal mask is needed here, since the cache holds only the current position and those before it, and there are no future positions to mask.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

位置嵌入用 `model.pos_emb(torch.tensor([pos]))`。忘记加位置或者位置编号错位，是这个练习中最常见的错误。

The positional embedding is `model.pos_emb(torch.tensor([pos]))`. Forgetting the position or shifting its index is the most common mistake in this exercise.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch


def attention_step(attn, x_new, cache):
    B, _, d = x_new.shape
    q, k, v = attn.qkv(x_new).chunk(3, dim=-1)                                    # each (B, 1, d)
    split = lambda t: t.view(B, 1, attn.n_head, attn.d_head).transpose(1, 2)      # (B, h, 1, d_head)
    q, k, v = split(q), split(k), split(v)
    if cache is not None:
        k = torch.cat([cache[0], k], dim=2)                                       # (B, h, t+1, d_head)
        v = torch.cat([cache[1], v], dim=2)
    scores = q @ k.transpose(-2, -1) / math.sqrt(attn.d_head)                     # (B, h, 1, t+1), no mask needed
    out = torch.softmax(scores, dim=-1) @ v                                       # (B, h, 1, d_head)
    out = out.transpose(1, 2).contiguous().view(B, 1, d)
    return attn.proj(out), (k, v)


@torch.no_grad()
def forward_step(model, token, pos, caches):
    x = model.tok_emb(token) + model.pos_emb(torch.tensor([pos]))                 # (B, 1, d)
    new_caches = []
    for block, cache in zip(model.blocks, caches):
        a, cache = attention_step(block.attn, block.ln1(x), cache)
        x = x + a
        x = x + block.mlp(block.ln2(x))
        new_caches.append(cache)
    return model.head(model.ln_f(x))[:, 0], new_caches


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    caches = [None] * len(model.blocks)
    for pos in range(idx.shape[1]):                                               # prefill, one token at a time
        logits, caches = forward_step(model, idx[:, pos:pos + 1], pos, caches)
    for i in range(max_new_tokens):
        nxt = logits.argmax(-1, keepdim=True)                                     # (B, 1)
        idx = torch.cat([idx, nxt], dim=1)
        if i < max_new_tokens - 1:                                                # the last token needs no further pass
            logits, caches = forward_step(model, nxt, idx.shape[1] - 1, caches)
    return idx
```

</details>

## 2. 测量首 token 延迟与生成速度 / Measure time to first token and generation speed

在 GPT-2 上分别实现不使用缓存和使用缓存的贪心生成，并记录每个 token 的耗时。测试会检查两种方式生成的 token 相同、缓存的实际大小与公式一致，并在 50、200、800 三种提示词长度下比较首 token 延迟和每秒生成的 token 数。时间的断言留了较大的余量，具体数值因机器而异。

Implement greedy generation on GPT-2 without and with the cache and record the time of every token. The tests check that both ways generate the same tokens and that the actual size of the cache agrees with the formula, and compare time to first token and tokens per second for prompts of 50, 200 and 800 tokens. The timing assertions leave a wide margin, and concrete numbers vary by machine.

需要联网下载模型。/ Downloads a model.

In [ ]:
import inspect
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def model_step(model, input_ids, cache=None):
    """Run the model and return (logits, cache). Nothing to change here.

    Hugging Face renamed the argument from `past_key_values` to `cache`; this works with both.
    """
    name = "cache" if "cache" in inspect.signature(model.forward).parameters else "past_key_values"
    out = model(input_ids, use_cache=True, **{name: cache})
    return out.logits, out[name]


def cache_tensors(cache):
    """The list of (keys, values) tensors, one pair per layer. Nothing to change here."""
    if hasattr(cache, "layers"):
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, "key_cache"):
        return list(zip(cache.key_cache, cache.value_cache))
    return [(layer[0], layer[1]) for layer in cache]


@torch.no_grad()
def generate_no_cache(model, input_ids, max_new_tokens):
    """Greedy decoding that runs the full sequence through the model at every step.

    Returns (ids (1, T + max_new_tokens), times), where times[i] is the wall-clock time in seconds
    spent producing new token i. Measure with time.perf_counter().
    """
    # TODO 1
    raise NotImplementedError


@torch.no_grad()
def generate_with_cache(model, input_ids, max_new_tokens):
    """Greedy decoding with the KV cache.

    Prefill: one call of model_step on the whole prompt gives the first new token and the cache.
    Decode:  every later call receives only the newest token, shape (1, 1), plus the cache.
    Returns (ids, times, cache), with ids and times as in generate_no_cache.
    """
    # TODO 2
    raise NotImplementedError


def latency_summary(times):
    """The two numbers serving systems report.

    ttft:              time to first token, times[0]
    tokens_per_second: decoding speed after the first token, (number of later tokens) / (their total time)
    Returns a dict with these two keys.
    """
    # TODO 3
    raise NotImplementedError


def kv_cache_bytes(n_layer, n_head, d_head, seq_len, bytes_per_number=4, batch=1):
    """Memory of the cache: one key and one value vector per layer, head and position."""
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
text = " ".join(["The quick brown fox jumps over the lazy dog."] * 90)
full = tok(text, return_tensors="pt").input_ids[:, :800]
N = 12

assert latency_summary([0.5, 0.1, 0.1, 0.2]) == {"ttft": 0.5, "tokens_per_second": 3 / 0.4}
assert kv_cache_bytes(12, 12, 64, 1024) == 2 * 12 * 12 * 64 * 1024 * 4            # about 75 MB for GPT-2 small at full context

rows = []
for T in (50, 200, 800):
    prompt = full[:, :T]
    a, ta = generate_no_cache(model, prompt, N)
    b, tb, cache = generate_with_cache(model, prompt, N)
    # 1. The cache changes the cost, not the result.
    assert a.shape == (1, T + N) and torch.equal(a, b), "both methods must produce the same tokens"
    assert len(ta) == N and len(tb) == N
    # 2. The cache holds every position that has passed through the model: the prompt and all but the last new token.
    pairs = cache_tensors(cache)
    assert len(pairs) == 12 and pairs[0][0].shape == (1, 12, T + N - 1, 64)
    measured = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in pairs)
    assert measured == kv_cache_bytes(12, 12, 64, T + N - 1), "the formula should match the real cache exactly"
    rows.append((T, latency_summary(ta), latency_summary(tb), measured))

# 3. Timing. The thresholds are loose because wall-clock time varies between machines.
no800, yes800 = rows[2][1], rows[2][2]
assert yes800["tokens_per_second"] > 3 * no800["tokens_per_second"], "with an 800-token prompt the cache should speed decoding up several times"
assert rows[2][2]["ttft"] > 2 * rows[0][2]["ttft"], "time to first token grows with the prompt length"
decode = [r[2]["tokens_per_second"] for r in rows]
assert max(decode) < 4 * min(decode), "with the cache, decoding speed depends only weakly on the prompt length"

print("m52_latency: all tests passed")
print("  prompt   time to first token        decoding speed (tokens/s)      cache size")
print("  tokens   no cache   with cache       no cache   with cache")
for T, s0, s1, size in rows:
    print(f"  {T:5d}   {s0['ttft']:7.3f}s   {s1['ttft']:7.3f}s      {s0['tokens_per_second']:8.1f}   {s1['tokens_per_second']:8.1f}        {size / 1e6:6.1f} MB")

<details><summary><b>提示 1 / Hint 1</b></summary>

prefill 是 `logits, cache = model_step(model, input_ids)`，之后每一步是 `model_step(model, next_id.view(1, 1), cache)`。

Prefill is `logits, cache = model_step(model, input_ids)`, and every later step is `model_step(model, next_id.view(1, 1), cache)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

每个 token 的计时应当包含模型的前向计算和取最大值两步。

The timing of each token should cover both the forward pass of the model and taking the maximum.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import inspect
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def model_step(model, input_ids, cache=None):
    """Run the model and return (logits, cache). Nothing to change here.

    Hugging Face renamed the argument from `past_key_values` to `cache`; this works with both.
    """
    name = "cache" if "cache" in inspect.signature(model.forward).parameters else "past_key_values"
    out = model(input_ids, use_cache=True, **{name: cache})
    return out.logits, out[name]


def cache_tensors(cache):
    """The list of (keys, values) tensors, one pair per layer. Nothing to change here."""
    if hasattr(cache, "layers"):
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, "key_cache"):
        return list(zip(cache.key_cache, cache.value_cache))
    return [(layer[0], layer[1]) for layer in cache]


@torch.no_grad()
def generate_no_cache(model, input_ids, max_new_tokens):
    times = []
    for _ in range(max_new_tokens):
        t0 = time.perf_counter()
        next_id = model(input_ids).logits[0, -1].argmax()
        input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
        times.append(time.perf_counter() - t0)
    return input_ids, times


@torch.no_grad()
def generate_with_cache(model, input_ids, max_new_tokens):
    times = []
    t0 = time.perf_counter()
    logits, cache = model_step(model, input_ids)                 # prefill: the whole prompt in one pass
    next_id = logits[0, -1].argmax()
    input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
    times.append(time.perf_counter() - t0)
    for _ in range(max_new_tokens - 1):
        t0 = time.perf_counter()
        logits, cache = model_step(model, next_id.view(1, 1), cache)   # decode: only the newest token
        next_id = logits[0, -1].argmax()
        input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
        times.append(time.perf_counter() - t0)
    return input_ids, times, cache


def latency_summary(times):
    ttft = times[0]
    decode = times[1:]
    return {"ttft": ttft, "tokens_per_second": len(decode) / sum(decode)}


def kv_cache_bytes(n_layer, n_head, d_head, seq_len, bytes_per_number=4, batch=1):
    return 2 * n_layer * batch * n_head * seq_len * d_head * bytes_per_number
```

</details>

## 3. 估算缓存的收益与内存 / Estimate the gain and the memory of the cache

在纸上完成。
(a) 用归纳法说明，因果掩码下第 $i$ 个位置在任意一层的表示只依赖于 $x_1, \dots, x_i$。
(b) 不使用缓存时，从 1 个 token 生成到 $T$ 个 token，模型一共处理了多少个位置？使用缓存时呢？$T = 1000$ 时两者相差多少倍？
(c) 一个模型有 24 层，$d = 2048$，以 16 位浮点数保存。上下文长度为 4096 时，单个序列的缓存占多少内存？一块 24 GB 的显卡在模型权重占去 8 GB 之后，最多能同时容纳多少个这样的序列？
(d) 一个请求的提示词有 2000 个 token，输出 500 个 token。prefill 的速度是每秒 10000 个 token，decode 的速度是每秒 50 个 token。首 token 延迟和总时间各是多少？哪个阶段占主要部分？

Work on paper.
(a) Show by induction that under the causal mask the representation of position $i$ in any layer depends only on $x_1, \dots, x_i$.
(b) Without the cache, how many positions does the model process in total when generating from 1 token up to $T$ tokens? And with the cache? By what factor do the two differ for $T = 1000$?
(c) A model has 24 layers and $d = 2048$ and is stored in 16-bit floating point. How much memory does the cache of one sequence take at a context length of 4096? How many such sequences fit at most on a 24 GB GPU after the model weights take 8 GB?
(d) A request has a prompt of 2000 tokens and an output of 500 tokens. Prefill runs at 10000 tokens per second and decode at 50 tokens per second. What are the time to first token and the total time, and which phase dominates?

<details><summary><b>提示 1 / Hint 1</b></summary>

(b) $1 + 2 + \dots + T = T(T+1)/2$。

(b) $1 + 2 + \dots + T = T(T+1)/2$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 字节数为 $2 \times L \times T \times d \times b$。

(c) The number of bytes is $2 \times L \times T \times d \times b$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 第 0 层的 $h_i^{(0)}$ 是 $x_i$ 的嵌入加位置编码，只依赖 $x_i$。假设第 $\ell - 1$ 层的 $h_j^{(\ell-1)}$ 只依赖 $x_1, \dots, x_j$。第 $\ell$ 层中，位置 $i$ 的注意力在因果掩码下只读取 $h_1^{(\ell-1)}, \dots, h_i^{(\ell-1)}$，前馈层和归一化只作用于位置 $i$ 自身，所以 $h_i^{(\ell)}$ 只依赖 $x_1, \dots, x_i$。

(b) 不使用缓存时为 $1 + 2 + \dots + T = T(T+1)/2$，$T = 1000$ 时是 500500 个位置。使用缓存时每个位置只处理一次，共 $T = 1000$ 个。两者相差约 500 倍。

(c) $2 \times 24 \times 4096 \times 2048 \times 2 = 805{,}306{,}368$ 字节，约 0.81 GB（0.75 GiB）。剩余 16 GB 最多容纳 $16 / 0.81 \approx 19$ 个序列。实际系统还需要为中间计算留出空间，能容纳的数量更少。

(d) 首 token 延迟约为 $2000 / 10000 = 0.2$ 秒。生成 500 个 token 需要 $500 / 50 = 10$ 秒，总时间约 10.2 秒。尽管输入的 token 数是输出的 4 倍，绝大部分时间花在 decode 上。

---

(a) In layer 0, $h_i^{(0)}$ is the embedding of $x_i$ plus the positional encoding and depends only on $x_i$. Assume $h_j^{(\ell-1)}$ in layer $\ell - 1$ depends only on $x_1, \dots, x_j$. In layer $\ell$ the attention of position $i$ reads only $h_1^{(\ell-1)}, \dots, h_i^{(\ell-1)}$ under the causal mask, and the feed-forward layer and normalization act on position $i$ alone, so $h_i^{(\ell)}$ depends only on $x_1, \dots, x_i$.

(b) Without the cache it is $1 + 2 + \dots + T = T(T+1)/2$, which is 500500 positions for $T = 1000$. With the cache every position is processed once, $T = 1000$ in total. The factor is about 500.

(c) $2 \times 24 \times 4096 \times 2048 \times 2 = 805{,}306{,}368$ bytes, about 0.81 GB (0.75 GiB). The remaining 16 GB hold at most $16 / 0.81 \approx 19$ sequences. A real system also needs room for intermediate computation and holds fewer.

(d) The time to first token is about $2000 / 10000 = 0.2$ seconds. Generating 500 tokens takes $500 / 50 = 10$ seconds, for a total of about 10.2 seconds. Although the input has 4 times as many tokens as the output, nearly all of the time goes into decode.

</details>